# 🛠️ Chapter 03-03b: Hands-On — LLM That Searches the Web & Answers

---

## Time to Give Your LLM Superpowers! 🦸

In this notebook, you'll build an LLM that can actually DO things — search the web, perform calculations, and more.

By the end:
- ✅ Created output parsers (JSON, Pydantic)
- ✅ Defined custom tools with @tool
- ✅ Bound tools to models with .bind_tools()
- ✅ Handled ToolCalls and ToolMessages
- ✅ Built a web-searching LLM assistant

Let's build! 🚀

---

## 📦 Setup

In [ ]:
# !pip install langchain-core langchain-openai python-dotenv

In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ Ready!" if os.getenv("OPENAI_API_KEY") else "❌ Set OPENAI_API_KEY in .env")

---

## 🧪 Exercise 1: Output Parsers

In [ ]:
# StrOutputParser — extract clean text

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer in one sentence."),
    ("human", "{question}")
])

# Without parser: get AIMessage object
chain_raw = prompt | llm
result_raw = chain_raw.invoke({"question": "What is Python?"})
print(f"Without parser: {type(result_raw).__name__} → {result_raw.content}")

# With StrOutputParser: get clean string
chain_str = prompt | llm | StrOutputParser()
result_str = chain_str.invoke({"question": "What is Python?"})
print(f"With parser:    {type(result_str).__name__} → {result_str}")

In [ ]:
# JsonOutputParser — extract structured dict

json_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the text and return valid JSON with:
{{"sentiment": "positive/negative/neutral", "topics": ["list"], "word_count": number}}
Return ONLY the JSON."""),
    ("human", "{text}")
])

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({"text": "Python is an amazing language for AI and machine learning!"})
print(f"Type: {type(result).__name__}")
print(f"Result: {result}")
print(f"Sentiment: {result['sentiment']}")
print(f"Topics: {result['topics']}")

---

## 🧪 Exercise 2: Creating Custom Tools

In [ ]:
from langchain_core.tools import tool

# Custom tool 1: Calculator
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Use this for any math calculation.
    Input should be a valid Python math expression like '2 + 3 * 4'."""
    try:
        # Using eval safely for math expressions only
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Only basic math operations are allowed"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Custom tool 2: Word counter
@tool
def count_words(text: str) -> str:
    """Count the number of words in a text. Use this when asked about text length."""
    count = len(text.split())
    return f"The text contains {count} words."

# Custom tool 3: Simulated web search
@tool
def search_web(query: str) -> str:
    """Search the internet for current information. Use this when you need real-time data."""
    # In production, this would call a real search API (Tavily, SerpAPI, etc.)
    return f"Search results for '{query}': [Simulated] Latest information about {query} as of 2024."

# Inspect our tools
tools = [calculate, count_words, search_web]
for t in tools:
    print(f"🔧 {t.name}: {t.description[:60]}...")
    print(f"   Args: {t.args}\n")

---

## 🧪 Exercise 3: Binding Tools to a Model

In [ ]:
# Bind tools to the model
llm_with_tools = llm.bind_tools(tools)

# Test: Ask a question that needs a tool
result = llm_with_tools.invoke("What is 1547 * 389?")

print("📤 LLM Response:")
print(f"   Content: '{result.content}'")
print(f"   Tool Calls: {result.tool_calls}")

# The LLM decided to call 'calculate'!
if result.tool_calls:
    tc = result.tool_calls[0]
    print(f"\n🔧 Tool Call:")
    print(f"   Name: {tc['name']}")
    print(f"   Args: {tc['args']}")
    print(f"   ID:   {tc['id']}")

In [ ]:
# Test: Ask something that DOESN'T need a tool
result = llm_with_tools.invoke("What is Python?")

print("📤 LLM Response (no tool needed):")
print(f"   Content: '{result.content[:100]}...'")
print(f"   Tool Calls: {result.tool_calls}")
print("\n💡 The model smartly chose NOT to use any tool!")

---

## 🧪 Exercise 4: Full Tool Execution Loop

In [ ]:
# The full loop: User → LLM → ToolCall → Execute → ToolMessage → LLM → Final Answer

# Map tool names to functions
tool_map = {t.name: t for t in tools}

def run_with_tools(question: str) -> str:
    """Run a question through the LLM with tool support."""
    messages = [HumanMessage(content=question)]
    
    # Step 1: Ask the LLM
    response = llm_with_tools.invoke(messages)
    messages.append(response)
    
    # Step 2: If tool calls, execute them
    if response.tool_calls:
        for tc in response.tool_calls:
            print(f"  🔧 Calling: {tc['name']}({tc['args']})")
            
            # Execute the tool
            tool_fn = tool_map[tc['name']]
            tool_result = tool_fn.invoke(tc['args'])
            
            print(f"  📥 Result: {tool_result}")
            
            # Add as ToolMessage
            messages.append(ToolMessage(
                content=str(tool_result),
                tool_call_id=tc['id']
            ))
        
        # Step 3: Get final answer
        final = llm_with_tools.invoke(messages)
        return final.content
    else:
        return response.content

# Test it!
questions = [
    "What is 2847 * 193 + 456?",
    "How many words are in: 'The quick brown fox jumps over the lazy dog'",
    "What is the capital of France?",  # No tool needed
]

for q in questions:
    print(f"\n❓ {q}")
    answer = run_with_tools(q)
    print(f"✅ {answer}")
    print("-" * 60)

---

## 🚀 Build: Web-Searching LLM Assistant

A mini-agent that decides when to search and when to answer directly.

In [ ]:
# Enhanced assistant with multiple tools and system prompt

@tool
def get_current_time() -> str:
    """Get the current date and time. Use when asked about today's date or current time."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

all_tools = [calculate, count_words, search_web, get_current_time]
all_tool_map = {t.name: t for t in all_tools}

assistant_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(all_tools)

def smart_assistant(question: str) -> str:
    """A smart assistant that uses tools when needed."""
    system = """You are a helpful AI assistant with access to tools.
Use tools when you need real data, calculations, or current information.
Answer directly when you can from your training data."""
    
    messages = [
        {"role": "system", "content": system},
        HumanMessage(content=question)
    ]
    
    response = assistant_llm.invoke(messages)
    messages.append(response)
    
    # Handle tool calls (potentially multiple)
    while response.tool_calls:
        for tc in response.tool_calls:
            print(f"  🔧 Using: {tc['name']}")
            tool_result = all_tool_map[tc['name']].invoke(tc['args'])
            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tc['id']))
        
        response = assistant_llm.invoke(messages)
        messages.append(response)
    
    return response.content

# Test the assistant
test_questions = [
    "What's 15% tip on a $87.50 dinner bill?",
    "What time is it right now?",
    "Search for the latest news about AI agents",
    "Explain recursion in one sentence",  # No tool needed
]

for q in test_questions:
    print(f"\n❓ {q}")
    answer = smart_assistant(q)
    print(f"💬 {answer}")
    print("═" * 60)

---

## ✍️ Practice: Build a Unit Converter Tool

**Challenge:** Create a tool that converts between units (km/miles, kg/lbs, celsius/fahrenheit).

In [ ]:
# YOUR CODE HERE

# TODO: Create a @tool function 'convert_units'
#   Parameters: value (float), from_unit (str), to_unit (str)
#   Support: km↔miles, kg↔lbs, celsius↔fahrenheit

# TODO: Bind to LLM and test with:
#   "Convert 100 km to miles"
#   "What is 72°F in Celsius?"
#   "How many lbs is 80 kg?"

<details>
<summary>💡 Click to see solution</summary>

```python
@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between units. Supports: km/miles, kg/lbs, celsius/fahrenheit."""
    conversions = {
        ("km", "miles"): lambda v: v * 0.621371,
        ("miles", "km"): lambda v: v * 1.60934,
        ("kg", "lbs"): lambda v: v * 2.20462,
        ("lbs", "kg"): lambda v: v * 0.453592,
        ("celsius", "fahrenheit"): lambda v: (v * 9/5) + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5/9,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        result = conversions[key](value)
        return f"{value} {from_unit} = {result:.2f} {to_unit}"
    return f"Unsupported conversion: {from_unit} to {to_unit}"

converter_llm = llm.bind_tools([convert_units])
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 StrOutputParser for text, JsonOutputParser for dicts.          │
├─────────────────────────────────────────────────────────────────────┤
│  📌 @tool + docstring = custom tool. Description is critical!      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 .bind_tools() attaches tools to a model.                       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 LLM returns ToolCall → you execute → ToolMessage → LLM final. │
├─────────────────────────────────────────────────────────────────────┤
│  📌 while response.tool_calls loop = manual agent!                 │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Agents (Ch. 6) automate this entire loop.                      │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Embeddings & Vector Stores!

Chapter 4: Deep dive into embeddings, vector databases, and semantic search. The foundation of RAG!

---

### 🎉 Your LLM Can Now DO Things!
Tools, function calling, output parsing — you've built an LLM that acts, not just talks! 🚀